# ML01 · 数据与张量基础

机器学习的第一步：学会把数据整理成模型看得懂的形状，并理解张量（tensor）与标准化（standardization）的基本直觉。

## 学习目标
- 能分辨特征（feature）与标签（label），并说出数据矩阵的形状含义（样本数、特征数）。
- 能用 numpy 数组创建、查看与操作数据，掌握形状（shape）与数据类型（dtype）。
- 理解特征标准化的动机，并用 z-score 把特征转成可比较的尺度。
- 认识张量的概念，理解它与 numpy 数组的关系与差异。
- 能把一批数据切成训练集（training set）与测试集（test set），并说明为何要这样切。

## 数据的形状：特征与标签

机器学习的输入几乎都是一张二维表：一列（row）是一个样本（sample），一栏（column）是一个特征（feature）。
- 特征：用来描述样本的输入数值，例如一栋建筑的楼高、面积、屋龄。
- 标签（label）：我们想预测的答案，例如这栋是不是住宅。

为什么先学这个：创建「一列一样本、一栏一特征」的心智模型后，后面所有数据都能套进这个形状。

形状：特征矩阵为 (样本数 N, 特征数 D)，标签为长度 N 的矢量。

In [ ]:
# 概念：把数据排成「N 个样本 × D 个特征」的二维矩阵，标签是长度 N 的一维矢量
import numpy as np

# 造一份假建筑数据：每栋有 楼高(公尺) / 面积(平方公尺) / 屋龄(年) 三个特征
features = np.array([
    [12.0, 85.0, 5.0],
    [30.0, 120.0, 12.0],
    [9.0, 60.0, 25.0],
    [45.0, 200.0, 3.0],
    [18.0, 95.0, 8.0],
])

# 标签：1 代表住宅、0 代表非住宅（与上面每一列一一对应）
labels = np.array([1, 1, 0, 1, 0])

print("特征矩阵（每列一栋建筑）:")
print(features)
print("标签矢量:", labels)
# shape 读作 (样本数, 特征数)；这里是 5 栋、每栋 3 个特征
print("特征 shape:", features.shape, "  标签 shape:", labels.shape)

## numpy 数组复习

numpy 数组（ndarray）是后面所有运算的底层容器。它支持索引（indexing）与切片（slicing），并能用矢量化运算（vectorization）一次处理整栏，取代逐笔 for 循环。

为什么重要：矢量化既快又贴近数学写法；广播（broadcasting）让「数组 + 单一数字」自动套用到每个元素。

形状：`数组[列索引, 栏索引]`，用 `:` 代表整列或整栏。

In [ ]:
# 概念：用切片依列／栏取值，用矢量化与广播一次处理整栏
import numpy as np

features = np.array([
    [12.0, 85.0, 5.0],
    [30.0, 120.0, 12.0],
    [9.0, 60.0, 25.0],
    [45.0, 200.0, 3.0],
    [18.0, 95.0, 8.0],
])

heights = features[:, 0]      # 取第 0 栏（所有楼高）；冒号代表「所有列」
first_three = features[:3]    # 取前三栋（前三列），等同 features[0:3]

print("所有楼高:", heights)
print("前三栋:\n", first_three)

# 广播：整栏一次加上 1（每栋楼高 +1 公尺），不需要写循环
print("楼高各加 1 公尺:", heights + 1)
# 矢量化：整栏面积乘以 0.9（例如打九折的可用面积）
print("面积打九折:", features[:, 1] * 0.9)

## 形状 shape 与数据类型 dtype

shape 描述数组每个轴（axis）的长度；`reshape` 可在不改变数据的前提下换一个形状。数据类型（dtype）决定数值是整数还是浮点数。

为什么重要：形状对不上是初学者最常见的错误来源；整数运算会把小数截掉，标准化等计算需要浮点数。养成随手检查 shape 与 dtype 的习惯。

形状：`数组.reshape(列数, 栏数)`，其中一个维度可用 `-1` 让 numpy 自动推算。

In [ ]:
# 概念：reshape 改形状（数据不变），以及 int 与 float 的差别与类型转换 cast
import numpy as np

# 造一维容积率数据（6 栋）
far = np.array([2.25, 3.0, 1.8, 4.5, 2.1, 3.6])
print("原始 shape:", far.shape)             # (6,) 是一维

col_vector = far.reshape(-1, 1)              # -1 让 numpy 自动算列数，变成 (6, 1) 栏矢量
print("reshape 后 shape:", col_vector.shape)

# 整数型楼层数
floors_int = np.array([3, 7, 5, 12, 4])
print("整数楼层 dtype:", floors_int.dtype)

# 注意：整数除法会截掉小数，例如平均每层用的概念计算会失真
print("整数相除（会截断）:", floors_int / 2)  # numpy 除法本身会升成 float，但若用 int 累积就会掉精度

floors_float = floors_int.astype(float)      # cast 成浮点数，保留小数
print("转成浮点 dtype:", floors_float.dtype, " 内容:", floors_float)

## 特征标准化 standardization

不同特征的数量级常常差很多：楼高是十几公尺、面积是上百平方公尺。若不处理，模型容易被大数值特征主导。

特征标准化（standardization）用 z-score 把每个特征转成「平均 0、标准差 1」的尺度，让每个特征站在同一条起跑在线。要逐栏（依特征）计算统计量。

公式：z = (x − 平均) / 标准差。

In [ ]:
# 概念：逐栏做 z-score 标准化，并比较标准化前后每栏的平均与分布范围
import numpy as np

features = np.array([
    [12.0, 85.0, 5.0],
    [30.0, 120.0, 12.0],
    [9.0, 60.0, 25.0],
    [45.0, 200.0, 3.0],
    [18.0, 95.0, 8.0],
])

# axis=0 代表「沿着列方向往下压」，得到每一栏（每个特征）的统计量
col_mean = features.mean(axis=0)   # 每栏平均
col_std = features.std(axis=0)     # 每栏标准差

standardized = (features - col_mean) / col_std   # 广播：每栏各自减平均、除标准差

print("标准化前 每栏平均:", np.round(col_mean, 2))
print("标准化前 每栏范围:", np.round(features.max(axis=0) - features.min(axis=0), 2))
print("标准化后 每栏平均:", np.round(standardized.mean(axis=0), 6))   # 应接近 0
print("标准化后 每栏标准差:", np.round(standardized.std(axis=0), 6))  # 应接近 1
# 技巧：标准差可能为 0（整栏相同）会造成除以 0；实务上会加一个很小的数避免
print("标准化后数据:\n", np.round(standardized, 3))

## 张量 tensor 入门

张量（tensor）就是「可以放在更高维、可被深度学习框架自动运算的多维数组」。它和 numpy 数组几乎是同一回事，只是多了之后会用到的能力（如自动微分的容器）。

用阶（rank）描述维度：纯量（scalar）是 0 阶、矢量（vector）是 1 阶、矩阵（matrix）是 2 阶、再上去就是高维张量。张量同样有 shape 与 dtype，并能与 numpy 互转。

本节用 numpy 仿真张量的概念与互转；若已安装 PyTorch，原理完全相同。

In [ ]:
# 概念：用 rank（阶）区分纯量／矢量／矩阵，并示范张量与 numpy 的互转
import numpy as np

scalar = np.array(7.0)                 # 0 阶：一个数
vector = np.array([1.0, 2.0, 3.0])     # 1 阶：一排数
matrix = np.array([[1.0, 2.0],
                   [3.0, 4.0]])        # 2 阶：一张表

# ndim 就是阶 rank（有几个轴）
print("纯量 rank:", scalar.ndim, " shape:", scalar.shape)
print("矢量 rank:", vector.ndim, " shape:", vector.shape)
print("矩阵 rank:", matrix.ndim, " shape:", matrix.shape)

# 概念：把 numpy 特征矩阵「当成张量」查看 shape 与 dtype，再转回 numpy 确认一致
standardized = (matrix - matrix.mean(axis=0)) / matrix.std(axis=0)
tensor_like = np.asarray(standardized, dtype=np.float32)   # 框架多用 float32，这里用它仿真张量
print("张量 shape:", tensor_like.shape, " dtype:", tensor_like.dtype)

back_to_numpy = np.asarray(tensor_like)                    # 转回 numpy
print("转回 numpy 后数据是否一致:", np.allclose(back_to_numpy, standardized))

## 训练集与测试集

模型要用「没看过的数据」来评估才公平。我们把数据切成训练集（training set）与测试集（test set）：用训练集学习、用测试集检验。

切分前先随机打乱（shuffle），避免数据原本的排序影响结果；常见比例是八比二。为什么不能拿测试数据训练：那等于先看过答案，评估会过度乐观。

形状：打乱索引后依比例切片，特征与标签要用「同一组索引」对齐。

In [ ]:
# 概念：先打乱再依八比二切成训练集与测试集，特征与标签用同一组索引对齐
import numpy as np

rng = np.random.default_rng(0)   # 固定乱数种子，让每次运行结果一致（方便对照）

# 造 10 栋建筑、各 3 个特征的假数据
n_samples = 10
features = rng.uniform(low=[5, 50, 0], high=[60, 250, 40], size=(n_samples, 3))
labels = rng.integers(low=0, high=2, size=n_samples)

indices = rng.permutation(n_samples)   # 打乱后的列索引
split = int(n_samples * 0.8)           # 八比二的切点
train_idx, test_idx = indices[:split], indices[split:]

# 注意：特征与标签必须用同一组索引切，否则样本与答案会错位
X_train, y_train = features[train_idx], labels[train_idx]
X_test, y_test = features[test_idx], labels[test_idx]

print("训练集样本数:", X_train.shape[0], " 特征 shape:", X_train.shape)
print("测试集样本数:", X_test.shape[0], " 特征 shape:", X_test.shape)
print("训练标签:", y_train, " 测试标签:", y_test)

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）建筑特征表的形状检查（集成：特征与标签 + numpy 数组）
#   情境：你拿到一份自造的社区数据，每栋记录 楼高 与 面积 两个特征，并标记是否为公寓。
#   要求：
#     1. 用 numpy 自造一个 8×2 的特征矩阵，与一个长度 8 的标签矢量（0/1）。
#     2. 印出特征矩阵的 shape，并用切片取出所有楼高栏。
#   验收：应该看到特征 shape 为 (8, 2)、标签长度为 8，以及一条只含楼高的一维数组。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）把社区数据标准化并转成张量（集成：shape 与 dtype + 标准化 + 张量）
#   情境：同一份社区数据中，楼高以公尺、面积以平方公尺记录，两个特征数量级差很大。
#   要求：
#     1. 对特征矩阵逐栏做 z-score 标准化（用 axis=0 算每栏平均与标准差），确认每栏标准化后平均接近 0。
#     2. 把结果转成浮点张量（dtype 为 float32），印出张量的 shape 与 dtype，再转回 numpy 比对是否一致。
#   验收：应看到标准化后各栏平均接近 0、张量 dtype 为浮点、shape 与原矩阵相同，且转回 numpy 后数值一致。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）设计一条可重用的数据前处理流程（集成：标准化 + 训练测试切分 + 形状／张量）
#   情境：你要为一份较大的自造都市建筑数据（含 楼高、面积、容积率 三个特征）准备可喂入模型的训练与测试数据。
#   要求：
#     1. 自造 N（例如 20）栋数据，先打乱，再依八比二切成训练集与测试集。
#     2. 只用「训练集」算出每栏平均与标准差，并用同一组统计量去标准化训练集与测试集。
#        （思考：为何不能用测试集自己的统计量？）
#     3. 把两份标准化后的特征都转成浮点张量，印出各自的 shape 与 dtype。
#   验收：应看到训练与测试样本数约为八比二、两份特征栏数一致、皆为浮点张量，
#         并能说明用训练集统计量处理测试集是为了避免信息泄漏（data leakage）。
# TODO: 在这里完成


## 小结

- 机器学习的数据通常排成二维表：一列一个样本、一栏一个特征，特征矩阵 shape 为 (样本数, 特征数)，标签为长度 N 的矢量。
- numpy 数组是运算底层；用索引、切片与矢量化（搭配广播）取代循环，既快又贴近数学。
- 随手检查 shape 与 dtype：reshape 换形状不改数据，浮点与整数的差别会影响计算精度。
- 特征标准化用 z-score（减平均、除标准差）让各特征尺度一致，避免大数值特征主导模型。
- 张量与 numpy 数组几乎同源，可互转，并具备之后深度学习要用到的能力。
- 切训练集与测试集（并先打乱）是公平评估的基本功；切分与依训练集统计量处理测试集，能避免「背答案」与信息泄漏。